# DESCN: Deep Entire Space Cross Networks for Individual Treatment Effect Estimation


## 1. Causal Inference


### 1.1 Potential Outcomes & Counterfactuals

For each individual $i$ with covariates $x_i$, treatment $w_i \in \{0,1\}$:

- $Y_i(1)$ — outcome if treated ($w_i = 1$)
- $Y_i(0)$ — outcome if not treated ($w_i = 0$)

Only one is observed: $y_i = w_i Y_i(1) + (1-w_i)Y_i(0)$. The unobserved outcome is the **counterfactual**.

### 1.2 Key Estimands

| Estimand | Definition | Meaning |
|---|---|---|
| **ITE / CATE** | $\tau(x) = \mathbb{E}[Y(1)-Y(0) \mid X=x]$ | Treatment effect for individuals with features $x$ |
| **ATE** | $\mathbb{E}[Y(1)-Y(0)]$ | Average effect over whole population |
| **ATT** | $\mathbb{E}[Y(1)-Y(0) \mid W=1]$ | Average effect on the treated |

**Treated Response**: $\mu_1(x) = \mathbb{E}[Y \mid W=1, X=x]$  
**Control Response**: $\mu_0(x) = \mathbb{E}[Y \mid W=0, X=x]$  
**Propensity Score**: $\pi(x) = P(W=1 \mid X=x)$

ITE is recovered as $\tau(x) = \mu_1(x) - \mu_0(x)$.

### 1.3 Three Assumptions

1. **Consistency**: $y_i = Y_i(w_i)$ — no interference between individuals
2. **Ignorability**: $Y(1),Y(0) \perp\!\!\!\perp W \mid X$ — no unmeasured confounders
3. **Overlap**: $0 < \pi(x) < 1$ for all $x$ — everyone could receive either treatment

### 1.4 Two Core Problems

| Problem | Cause | Example |
|---|---|---|
| **Treatment Bias** | Confounding → treated & control distributions differ | Inactive users get vouchers, active users don't |
| **Sample Imbalance** | $|T| \ll |C|$ or $|T| \gg |C|$ | Vouchers given to only 5% of users |


---


## 2. DESCN Architecture

DESCN addresses **both** treatment bias and sample imbalance through two integrated components.

### 2.1 Entire Space Network (ESN)

Instead of learning $\mu_1$ only on treated samples and $\mu_0$ only on control, ESN connects them via propensity:

$$\text{ESTR} = P(Y, W=1 \mid X) = \mu_1 \cdot \pi$$
$$\text{ESCR} = P(Y, W=0 \mid X) = \mu_0 \cdot (1-\pi)$$

ESTR and ESCR are trained on **all** samples. A treated sample contributes to learning $\mu_0$ (via ESCR), and vice versa.  
ESN implicitly performs Inverse Probability Weighting: $ATE = \mathbb{E}[\text{ESTR}/\pi] - \mathbb{E}[\text{ESCR}/(1-\pi)]$.

**ESN loss**: $\mathcal{L}_{ESN} = \alpha\mathcal{L}_{\pi} + \beta_1\mathcal{L}_{ESTR} + \beta_0\mathcal{L}_{ESCR}$

### 2.2 X-network

Introduces a **Pseudo Treatment Effect** $\tau'$ as a bridge between TR and CR, operating in logit space:

$$\mu_1' = \sigma\big(\sigma^{-1}(\mu_0) + \sigma^{-1}(\tau')\big) \quad \text{— Cross Treated Response}$$
$$\mu_0' = \sigma\big(\sigma^{-1}(\mu_1) - \sigma^{-1}(\tau')\big) \quad \text{— Cross Control Response}$$

Logit-space operations are numerically stable, keep outputs in $[0,1]$, and magnify uplift signals near boundaries.

**X-network losses**: $\mathcal{L}_{CrossTR}$ (on treated) and $\mathcal{L}_{CrossCR}$ (on control)

### 2.3 Model Variants

All variants share the same backbone — different loss weights produce different models:

| Model | $\alpha$ (prpsy) | $\beta_1,\beta_0$ (ESTR,ESCR) | $\gamma_1,\gamma_0$ (CrossTR,CrossCR) | $\lambda$ (IPM) | TR/CR |
|---|---|---|---|---|---|
| **TARNet** | 0 | 0, 0 | 0, 0 | 0 | 1, 1 |
| **CFR (MMD)** | 0 | 0, 0 | 0, 0 | 0.1 | 1, 1 |
| **X-network** | 0 | 0, 0 | 2, 1 | 0 | 2, 2 |
| **DESCN** | 0.5 | 0.5, 1 | 1, 0.5 | 0 | 0, 0 |

### 2.4 Full DESCN Loss

$$\mathcal{L}_{DESCN} = \alpha\mathcal{L}_{\pi} + \beta_1\mathcal{L}_{ESTR} + \beta_0\mathcal{L}_{ESCR} + \gamma_1\mathcal{L}_{CrossTR} + \gamma_0\mathcal{L}_{CrossCR}$$

> TR and CR are trained through ESTR/ESCR in the entire space, not directly ($h_1,h_0$ weights = 0).


---


## 3. Implementation


### 3.1 Setup


In [1]:
import gc, os, random, warnings

import numpy as np
import tensorflow as tf
from tqdm import tqdm

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # suppress TF C++ INFO/WARN logs

warnings.filterwarnings('ignore', category=FutureWarning)
tf.get_logger().setLevel('ERROR')

# ---- Reproducibility ----
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything(2)

# ---- Runtime / CUDA visibility ----
gpu_devices = tf.config.list_physical_devices('GPU')
for gpu_device in gpu_devices:
    try:
        tf.config.experimental.set_memory_growth(gpu_device, True)
    except RuntimeError as error:
        print(f'Could not set memory growth for {gpu_device.name}: {error}')

build_info = tf.sysconfig.get_build_info()
print(f'TF {tf.__version__}  |  GPUs visible: {len(gpu_devices)}')
if gpu_devices:
    for gpu_index, gpu_device in enumerate(gpu_devices):
        try:
            gpu_details = tf.config.experimental.get_device_details(gpu_device)
        except Exception:
            gpu_details = {}
        gpu_name = gpu_details.get('device_name', gpu_device.name)
        print(f'  GPU {gpu_index}: {gpu_name}')
    print(f"  CUDA build: {build_info.get('cuda_version', 'unknown')}")
    print(f"  cuDNN build: {build_info.get('cudnn_version', 'unknown')}")
else:
    print('  Running on CPU. On RunPod, check nvidia-smi and tensorflow[and-cuda] install if this is unexpected.')


TF 2.17.1  |  GPUs visible: 1
  GPU 0: METAL
  CUDA build: unknown
  cuDNN build: unknown


### 3.2 
### a) Lazada Production Dataset

The paper provides a real-world E-commerce voucher dataset from Lazada (public, non-commercial use):
- **Train**: 927K samples, biased treatment assignment ~22% treated
- **Test**: 182K samples, randomized controlled trial (RCT)
- **83 features** (f0–f82), binary label, binary treatment


In [2]:
import zipfile, urllib.request
from pathlib import Path

DATA_PATH = Path('./data')
DATA_PATH.mkdir(exist_ok=True)

lazada_zip = DATA_PATH / 'lzd_data_public.zip'
lazada_dir = DATA_PATH / 'lzd_data_public'

if not lazada_dir.exists():
    url = 'https://www.dropbox.com/s/07r7592h9mfijsb/lzd_data_public.zip?dl=1'
    print(f'Downloading Lazada dataset ({url})...')
    urllib.request.urlretrieve(url, lazada_zip)
    
    print('Extracting...')
    with zipfile.ZipFile(lazada_zip, 'r') as f:
        f.extractall(DATA_PATH)
    lazada_zip.unlink()
    print('Done.')
else:
    print(f'Lazada dataset already exists at {lazada_dir}')


Lazada dataset already exists at data/lzd_data_public


In [3]:
def load_lazada_data():
    import pandas as pd
    
    train_df = pd.read_csv(lazada_dir / 'full_trainset.csv')
    test_df = pd.read_csv(lazada_dir / 'full_testset.csv')
    
    # 83 features: f0–f82
    feature_cols = [c for c in train_df.columns if c.startswith('f')]
    
    def df_to_dict(df):
        return {
            'x': df[feature_cols].values.astype(np.float32),
            't': df['is_treat'].values.astype(np.float32),
            'yf': df['label'].values.astype(np.float32),
            'ycf': None,        # no counterfactual in real data
            'mu0': None,
            'mu1': None,
            'tau': None,        # no ground truth ITE
            'e': np.zeros(len(df), dtype=np.float32),  # all 0 = biased (train) / RCT handled below
        }
    
    train_data = df_to_dict(train_df)
    test_data = df_to_dict(test_df)
    test_data['e'] = np.ones(len(test_df), dtype=np.float32)  # test is RCT
    
    print(f'Lazada train: {train_data["x"].shape[0]:,} samples, {train_data["x"].shape[1]} features')
    print(f'  Treated: {train_data["t"].sum():.0f} ({train_data["t"].mean()*100:.1f}%)')
    print(f'  Outcome rate (treated): {train_data["yf"][train_data["t"]==1].mean():.3f}')
    print(f'  Outcome rate (control): {train_data["yf"][train_data["t"]==0].mean():.3f}')
    print(f'Lazada test (RCT): {test_data["x"].shape[0]:,} samples')
    print(f'  Treated: {test_data["t"].sum():.0f} ({test_data["t"].mean()*100:.1f}%)')
    print(f'  Outcome rate (treated): {test_data["yf"][test_data["t"]==1].mean():.3f}')
    print(f'  Outcome rate (control): {test_data["yf"][test_data["t"]==0].mean():.3f}')
    
    return train_data, test_data

print(f"Loaded Lazada dataset from {lazada_dir}")

Loaded Lazada dataset from data/lzd_data_public


### b) Synthetic Data


In [4]:
# Load Lazada production dataset
training_data, test_data = load_lazada_data()

# ===== EDA: Quick data check before training =====
import pandas as pd

train_df = pd.DataFrame(training_data['x'], columns=[f'f{i}' for i in range(training_data['x'].shape[1])])
train_df['treatment'] = training_data['t']
train_df['outcome'] = training_data['yf']
train_df['is_rct'] = training_data['e']

print('\n--- Train set EDA ---')
print(f'Shape: {train_df.shape}')
print(f'Features: {train_df.columns[:5].tolist()} ... {train_df.columns[-5:].tolist()}')
print(f'\nTreatment distribution:')
print(f'  Treated (t=1): {train_df["treatment"].sum():.0f} ({train_df["treatment"].mean()*100:.1f}%)')
print(f'  Control (t=0): {(1-train_df["treatment"]).sum():.0f} ({(1-train_df["treatment"]).mean()*100:.1f}%)')
print(f'\nOutcome rate:')
print(f'  Overall:    {train_df["outcome"].mean():.4f}')
print(f'  Treated:    {train_df[train_df["treatment"]==1]["outcome"].mean():.4f}')
print(f'  Control:    {train_df[train_df["treatment"]==0]["outcome"].mean():.4f}')
print(f'  Uplift (naive): {train_df[train_df["treatment"]==1]["outcome"].mean() - train_df[train_df["treatment"]==0]["outcome"].mean():.4f}')
print(f'\nRCT flags in train: {train_df["is_rct"].sum():.0f} / {len(train_df)}')
print(f'\nFeature stats:')
print(f'  Mean:  {training_data["x"].mean():.4f}')
print(f'  Std:   {training_data["x"].std():.4f}')
print(f'  Min:   {training_data["x"].min():.4f}')
print(f'  Max:   {training_data["x"].max():.4f}')
print(f'  NaN:   {np.isnan(training_data["x"]).sum()}')

test_df = pd.DataFrame(test_data['x'], columns=[f'f{i}' for i in range(test_data['x'].shape[1])])
test_df['treatment'] = test_data['t']
test_df['outcome'] = test_data['yf']
test_df['is_rct'] = test_data['e']

print('\n--- Test set EDA (RCT) ---')
print(f'Shape: {test_df.shape}')
print(f'Treatment: {test_df["treatment"].sum():.0f} treated ({test_df["treatment"].mean()*100:.1f}%)')
print(f'Outcome rate: {test_df["outcome"].mean():.4f}')
print(f'  Treated: {test_df[test_df["treatment"]==1]["outcome"].mean():.4f}')
print(f'  Control: {test_df[test_df["treatment"]==0]["outcome"].mean():.4f}')
print(f'  True uplift (RCT): {test_df[test_df["treatment"]==1]["outcome"].mean() - test_df[test_df["treatment"]==0]["outcome"].mean():.4f}')
print(f'RCT flags in test: {test_df["is_rct"].sum():.0f} / {len(test_df)}')


Lazada train: 926,669 samples, 83 features
  Treated: 205480 (22.2%)
  Outcome rate (treated): 0.057
  Outcome rate (control): 0.009
Lazada test (RCT): 181,669 samples
  Treated: 94682 (52.1%)
  Outcome rate (treated): 0.037
  Outcome rate (control): 0.033

--- Train set EDA ---
Shape: (926669, 86)
Features: ['f0', 'f1', 'f2', 'f3', 'f4'] ... ['f81', 'f82', 'treatment', 'outcome', 'is_rct']

Treatment distribution:
  Treated (t=1): 205480 (22.2%)
  Control (t=0): 721189 (77.8%)

Outcome rate:
  Overall:    0.0199
  Treated:    0.0566
  Control:    0.0094
  Uplift (naive): 0.0472

RCT flags in train: 0 / 926669

Feature stats:
  Mean:  6.0248
  Std:   36.3228
  Min:   -7.3463
  Max:   365.0000
  NaN:   0

--- Test set EDA (RCT) ---
Shape: (181669, 86)
Treatment: 94682 treated (52.1%)
Outcome rate: 0.0352
  Treated: 0.0370
  Control: 0.0333
  True uplift (RCT): 0.0037
RCT flags in test: 181669 / 181669


In [5]:
# ==============================================================================
# Public Lazada dataset: 926,669 train, 181,669 test (RCT), ~22% treated in train
# Experiment protocol follows DESCN/DeepModels_real_data.ipynb:
#   5 experiment subsets, each using 90% of the biased train CSV.
# ==============================================================================

HYPERPARAMS = {
    'share_dim': 128,          # hidden units in shared layers
    'base_dim': 64,            # hidden units in each head
    'batch_size': 5000,
    'lr': 0.001,
    'l2': 0.001,               # weight decay
    'dropout': 0.1,            # dropout rate (do_rate=0.1)
    'normalize': 'divide',     # L2-normalize shared representation
    'use_batchnorm': True,      # Lazada public configs set BatchNorm1d: true
}

TRAIN_CONFIG = {
    'epochs': 5,                       # epochs inside each independent Production experiment
    'decay_rate': 0.95,                # LR decay for Production
    'decay_step_size': 1,
    'val_ratio': 0.2,
    'experiment_train_fraction': 0.9,  # public code uses train_test_split(..., test_size=0.9)
    'reweight_sample': True,
    'prediction_output_dir': 'results/lzd_real_tf',
}

EXPERIMENT_COUNT = 5                   # paper/code repeats each experiment 5 times for mean +/- s.e.

# ---- Per-model loss weights (from conf4models/lzd_real_data/) ----
# fmt: off
MODEL_CONFIGS = {
    'TARNet': {
        'prpsy_w': 0, 'escvr1_w': 0, 'escvr0_w': 0,
        'h1_w': 1, 'h0_w': 1,
        'mu1hat_w': 0, 'mu0hat_w': 0,
        'imb_dist_w': 0,
    },
    'CFR_MMD': {
        'prpsy_w': 0, 'escvr1_w': 0, 'escvr0_w': 0,
        'h1_w': 1, 'h0_w': 1,
        'mu1hat_w': 0, 'mu0hat_w': 0,
        'imb_dist_w': 0.1, 'imb_dist': 'mmd',
    },
    'X_network': {
        'prpsy_w': 0, 'escvr1_w': 0, 'escvr0_w': 0,
        'h1_w': 2, 'h0_w': 2,
        'mu1hat_w': 2, 'mu0hat_w': 1,
        'imb_dist_w': 0,
    },
    'DESCN': {
        'prpsy_w': 0.5, 'escvr1_w': 0.5, 'escvr0_w': 1.0,
        'h1_w': 0, 'h0_w': 0,
        'mu1hat_w': 1.0, 'mu0hat_w': 0.5,
        'imb_dist_w': 0, 'imb_dist': 'wass',
    },
}
# fmt: on

print('Configuration loaded. Models defined:', list(MODEL_CONFIGS.keys()))


Configuration loaded. Models defined: ['TARNet', 'CFR_MMD', 'X_network', 'DESCN']


### 3.3 Train / Validation / Test Split


In [6]:
from sklearn.model_selection import train_test_split

training_features = training_data['x']
training_outcomes = training_data['yf'].reshape(-1, 1)
training_treatments = training_data['t'].reshape(-1, 1)
training_randomized_flags = training_data['e'].reshape(-1, 1)

sample_count = len(training_features)
all_training_indices = np.arange(sample_count)

def to_tensor(array):
    return tf.constant(array, dtype=tf.float32)

features_test = to_tensor(test_data['x'])
outcomes_test = to_tensor(test_data['yf'].reshape(-1, 1))
treatments_test = to_tensor(test_data['t'].reshape(-1, 1))
randomized_flags_test = to_tensor(test_data['e'].reshape(-1, 1))
treatment_effect_test = None  # Lazada has no ground truth ITE

experiment_splits = []
validation_random_state = np.random.RandomState(2)
for experiment_index in range(EXPERIMENT_COUNT):
    _, experiment_indices = train_test_split(
        all_training_indices,
        test_size=TRAIN_CONFIG['experiment_train_fraction'],
        random_state=experiment_index,
        shuffle=True,
    )

    validation_count = int(len(experiment_indices) * TRAIN_CONFIG['val_ratio'])
    train_count = len(experiment_indices) - validation_count
    shuffled_experiment_indices = validation_random_state.permutation(experiment_indices)
    train_indices = shuffled_experiment_indices[:train_count]
    validation_indices = shuffled_experiment_indices[train_count:]

    experiment_splits.append({
        'experiment_indices': experiment_indices,
        'train_indices': train_indices,
        'validation_indices': validation_indices,
    })

def make_experiment_tensors(experiment_index):
    experiment_split = experiment_splits[experiment_index]
    train_indices = experiment_split['train_indices']
    validation_indices = experiment_split['validation_indices']

    return {
        'features_train': to_tensor(training_features[train_indices]),
        'outcomes_train': to_tensor(training_outcomes[train_indices]),
        'treatments_train': to_tensor(training_treatments[train_indices]),
        'randomized_flags_train': to_tensor(training_randomized_flags[train_indices]),
        'features_validation': to_tensor(training_features[validation_indices]),
        'outcomes_validation': to_tensor(training_outcomes[validation_indices]),
        'treatments_validation': to_tensor(training_treatments[validation_indices]),
        'randomized_flags_validation': to_tensor(training_randomized_flags[validation_indices]),
        'features_test': features_test,
        'outcomes_test': outcomes_test,
        'treatments_test': treatments_test,
        'randomized_flags_test': randomized_flags_test,
        'treatment_effect_test': treatment_effect_test,
    }

first_split = experiment_splits[0]
print(f'Prepared {EXPERIMENT_COUNT} paper-style experiment subsets from public Lazada train CSV.')
print(
    f'Each subset: {len(first_split["experiment_indices"]):,} samples '
    f'({TRAIN_CONFIG["experiment_train_fraction"]:.0%} of train CSV) '
    f'-> {len(first_split["train_indices"]):,} train / '
    f'{len(first_split["validation_indices"]):,} val'
)
print(f'Test RCT set reused for all experiments: {len(features_test):,} samples')
for experiment_index, experiment_split in enumerate(experiment_splits):
    train_treated_ratio = training_treatments[experiment_split['train_indices']].mean()
    validation_treated_ratio = training_treatments[experiment_split['validation_indices']].mean()
    print(
        f'  exp {experiment_index + 1}: '
        f'train treated={train_treated_ratio:.3f}, '
        f'val treated={validation_treated_ratio:.3f}'
    )


Prepared 5 paper-style experiment subsets from public Lazada train CSV.
Each subset: 834,003 samples (90% of train CSV) -> 667,203 train / 166,800 val
Test RCT set reused for all experiments: 181,669 samples
  exp 1: train treated=0.222, val treated=0.222
  exp 2: train treated=0.222, val treated=0.223
  exp 3: train treated=0.222, val treated=0.220
  exp 4: train treated=0.222, val treated=0.222
  exp 5: train treated=0.222, val treated=0.222


### 3.4 Model Components


In [7]:
from tensorflow.keras import layers

def torch_style_dense(units, activation=None):
    return layers.Dense(
        units,
        activation=activation,
        kernel_initializer=tf.keras.initializers.VarianceScaling(
            scale=1.0, mode='fan_in', distribution='untruncated_normal'
        ),
        bias_initializer='zeros',
    )

# ---- ShareNetwork: features -> shared representation h (L2-normalized) ----
class ShareNetwork(tf.keras.Model):
    def __init__(self, share_dim=256, base_dim=128, dropout=0.0, normalize='divide', use_batchnorm=False, **kwargs):
        super().__init__(**kwargs)
        self.normalize = normalize
        layers_list = []
        if use_batchnorm:
            layers_list.append(layers.BatchNormalization(momentum=0.9, epsilon=1e-5))
        layers_list.append(torch_style_dense(share_dim, activation='elu'))
        if dropout > 0:
            layers_list.append(layers.Dropout(rate=dropout))
        layers_list.append(torch_style_dense(share_dim, activation='elu'))
        if dropout > 0:
            layers_list.append(layers.Dropout(rate=dropout))
        layers_list.append(torch_style_dense(base_dim, activation='elu'))
        if dropout > 0:
            layers_list.append(layers.Dropout(rate=dropout))
        self.deep_network = tf.keras.Sequential(layers_list)

    def call(self, input_features, training=False):
        shared_representation = self.deep_network(input_features, training=training)
        if self.normalize == 'divide':
            representation_norm = tf.sqrt(
                tf.reduce_sum(tf.square(shared_representation), axis=1, keepdims=True) + 1e-9
            )
            shared_representation = shared_representation / representation_norm
        return shared_representation

# ---- Head: base_dim -> base_dim -> base_dim -> 1 (logit) ----
def make_head(dim=128, dropout=0.0, name='head'):
    layers_list = []
    layers_list.append(torch_style_dense(dim, activation='elu'))
    if dropout > 0:
        layers_list.append(layers.Dropout(rate=dropout))
    layers_list.append(torch_style_dense(dim, activation='elu'))
    if dropout > 0:
        layers_list.append(layers.Dropout(rate=dropout))
    layers_list.append(torch_style_dense(dim, activation='elu'))
    if dropout > 0:
        layers_list.append(layers.Dropout(rate=dropout))
    layers_list.append(torch_style_dense(1))
    return tf.keras.Sequential(layers_list, name=name)

# ---- DESCN / ESX: shared backbone + 4 heads ----
class DESCN(tf.keras.Model):
    """
    Deep Entire Space Cross Networks.
    
    Forward returns 12 tensors:
      propensity_logit, entire_space_treated_response, entire_space_control_response,
      pseudo_effect_logit, treated_response_logit, control_response_logit,
      propensity, treated_response_probability, control_response_probability,
      treated_response_head, control_response_head, shared_representation
    
    The final ITE prediction follows the original code: p_h1 - p_h0 (probability space).
    """
    def __init__(self, input_dim, share_dim=256, base_dim=128,
                 dropout=0.0, normalize='divide', use_batchnorm=False, **kwargs):
        super().__init__(**kwargs)
        self.share_network = ShareNetwork(share_dim, base_dim, dropout, normalize, use_batchnorm)
        self.propensity_head = make_head(base_dim, dropout, 'propensity')
        self.treated_response_head = make_head(base_dim, dropout, 'mu1')
        self.control_response_head = make_head(base_dim, dropout, 'mu0')
        self.pseudo_effect_head = make_head(base_dim, dropout, 'tau')
    
    def call(self, input_features, training=False):
        shared_representation = self.share_network(input_features, training=training)
        
        propensity_logit = self.propensity_head(shared_representation, training=training)
        treated_response_logit = self.treated_response_head(shared_representation, training=training)
        control_response_logit = self.control_response_head(shared_representation, training=training)
        pseudo_effect_logit = self.pseudo_effect_head(shared_representation, training=training)
        
        propensity = tf.clip_by_value(tf.nn.sigmoid(propensity_logit), 1e-3, 1 - 1e-3)
        treated_response_probability = tf.nn.sigmoid(treated_response_logit)
        control_response_probability = tf.nn.sigmoid(control_response_logit)
        
        entire_space_treated_response = propensity * treated_response_probability
        entire_space_control_response = (1.0 - propensity) * control_response_probability
        
        return (
            propensity_logit,
            entire_space_treated_response,
            entire_space_control_response,
            pseudo_effect_logit,
            treated_response_logit,
            control_response_logit,
            propensity,
            treated_response_probability,
            control_response_probability,
            treated_response_probability,
            control_response_probability,
            shared_representation,
        )

print('Model components defined.')


Model components defined.


### 3.5 Loss Functions & IPM Distances


In [8]:
# ---- BCE on probabilities ----
def binary_cross_entropy(labels, predictions):
    predictions = tf.clip_by_value(predictions, 1e-7, 1 - 1e-7)
    return -tf.reduce_mean(labels * tf.math.log(predictions) + (1 - labels) * tf.math.log(1 - predictions))

# ---- Weighted BCE on probabilities ----
def weighted_binary_cross_entropy(labels, predictions, sample_weight):
    predictions = tf.clip_by_value(predictions, 1e-7, 1 - 1e-7)
    per_sample_loss = -(labels * tf.math.log(predictions) + (1 - labels) * tf.math.log(1 - predictions))
    return tf.reduce_mean(sample_weight * per_sample_loss)

# ---- BCE on logits with class weight (for propensity) ----
def weighted_logit_binary_cross_entropy(labels, logits, positive_weight):
    per_sample_loss = tf.nn.sigmoid_cross_entropy_with_logits(labels=labels, logits=logits)
    sample_weight = labels * positive_weight + (1.0 - labels)
    return tf.reduce_mean(sample_weight * per_sample_loss)

def pairwise_squared_distances(left_features, right_features):
    left_squared_norm = tf.reduce_sum(tf.square(left_features), axis=1, keepdims=True)
    right_squared_norm = tf.reduce_sum(tf.square(right_features), axis=1, keepdims=True)
    squared_distances = left_squared_norm - 2.0 * tf.matmul(left_features, right_features, transpose_b=True)
    squared_distances += tf.transpose(right_squared_norm)
    return tf.maximum(squared_distances, 0.0)

def pairwise_euclidean_distances(left_features, right_features):
    return tf.sqrt(tf.maximum(pairwise_squared_distances(left_features, right_features), 1e-9))

def split_treated_control(shared_representations, treatments):
    treatment_vector = tf.reshape(treatments, (-1,))
    treated_indices = tf.where(treatment_vector > 0.5)[:, 0]
    control_indices = tf.where(treatment_vector < 0.5)[:, 0]
    return (
        tf.gather(shared_representations, treated_indices),
        tf.gather(shared_representations, control_indices),
    )

def energy_distance_between_groups(shared_representations, treatments):
    treated_representations, control_representations = split_treated_control(shared_representations, treatments)

    def compute_distance():
        cross_distance = tf.reduce_mean(pairwise_euclidean_distances(treated_representations, control_representations))
        treated_distance = tf.reduce_mean(pairwise_euclidean_distances(treated_representations, treated_representations))
        control_distance = tf.reduce_mean(pairwise_euclidean_distances(control_representations, control_representations))
        return tf.maximum(2.0 * cross_distance - treated_distance - control_distance, 0.0)

    has_both_groups = tf.logical_and(tf.shape(treated_representations)[0] > 0, tf.shape(control_representations)[0] > 0)
    return tf.cond(has_both_groups, compute_distance, lambda: tf.constant(0.0, dtype=shared_representations.dtype))

def sinkhorn_transport_cost(left_features, right_features, epsilon=0.05, iterations=30):
    cost_matrix = pairwise_squared_distances(left_features, right_features)
    left_count = tf.shape(left_features)[0]
    right_count = tf.shape(right_features)[0]
    log_left_weights = -tf.math.log(tf.cast(left_count, cost_matrix.dtype)) * tf.ones((left_count,), dtype=cost_matrix.dtype)
    log_right_weights = -tf.math.log(tf.cast(right_count, cost_matrix.dtype)) * tf.ones((right_count,), dtype=cost_matrix.dtype)
    log_kernel = -cost_matrix / epsilon
    log_u = tf.zeros_like(log_left_weights)
    log_v = tf.zeros_like(log_right_weights)

    for _ in range(iterations):
        log_u = log_left_weights - tf.reduce_logsumexp(log_kernel + log_v[tf.newaxis, :], axis=1)
        log_v = log_right_weights - tf.reduce_logsumexp(log_kernel + log_u[:, tf.newaxis], axis=0)

    transport_plan = tf.exp(log_u[:, tf.newaxis] + log_kernel + log_v[tf.newaxis, :])
    return tf.reduce_sum(transport_plan * cost_matrix)

def sinkhorn_distance_between_groups(shared_representations, treatments):
    treated_representations, control_representations = split_treated_control(shared_representations, treatments)

    def compute_distance():
        # Approximation of GeomLoss SamplesLoss(loss="sinkhorn", p=2, blur=0.05).
        cross_cost = sinkhorn_transport_cost(treated_representations, control_representations)
        treated_self_cost = sinkhorn_transport_cost(treated_representations, treated_representations)
        control_self_cost = sinkhorn_transport_cost(control_representations, control_representations)
        return tf.maximum(cross_cost - 0.5 * treated_self_cost - 0.5 * control_self_cost, 0.0)

    has_both_groups = tf.logical_and(tf.shape(treated_representations)[0] > 0, tf.shape(control_representations)[0] > 0)
    return tf.cond(has_both_groups, compute_distance, lambda: tf.constant(0.0, dtype=shared_representations.dtype))

# ---- IPM distances ----
def wasserstein_distance(shared_representations, treatments):
    return sinkhorn_distance_between_groups(shared_representations, treatments)

def mmd_distance(shared_representations, treatments):
    # Original code names this CFRmmd, but GeomLoss uses SamplesLoss(loss="energy").
    return energy_distance_between_groups(shared_representations, treatments)

def masked_binary_cross_entropy(labels, predictions, mask):
    return tf.cond(
        tf.reduce_any(mask),
        lambda: binary_cross_entropy(tf.boolean_mask(labels, mask), tf.boolean_mask(predictions, mask)),
        lambda: tf.constant(0.0, dtype=predictions.dtype),
    )

print('Loss functions defined.')


Loss functions defined.


### 3.6 Evaluation


In [9]:
from sklift.metrics import qini_auc_score

def evaluate(
    model,
    features,
    outcomes,
    treatments,
    randomized_flags,
    treatment_effect_true=None,
    weights=None,
    compute_loss=True,
    compute_imbalance_loss=False,
):
    """Compute paper metrics and optional training-style losses.

    IPM imbalance loss is a training regularizer. Computing it on the full
    validation/test set creates huge pairwise matrices, so evaluation skips it
    unless explicitly requested.
    """
    loss_weights = weights or {}
    propensity_weight = loss_weights.get('prpsy_w', 0)
    entire_treated_weight = loss_weights.get('escvr1_w', 0)
    entire_control_weight = loss_weights.get('escvr0_w', 0)
    treated_response_weight = loss_weights.get('h1_w', 0)
    control_response_weight = loss_weights.get('h0_w', 0)
    cross_treated_weight = loss_weights.get('mu1hat_w', 0)
    cross_control_weight = loss_weights.get('mu0hat_w', 0)
    imbalance_weight = loss_weights.get('imb_dist_w', 0)
    imbalance_type = loss_weights.get('imb_dist', 'wass')
    reweight_sample = loss_weights.get('reweight_sample', True)
    
    (
        propensity_logit,
        entire_space_treated_response,
        entire_space_control_response,
        pseudo_effect_logit,
        treated_response_logit,
        control_response_logit,
        _,
        _,
        _,
        treated_response_head,
        control_response_head,
        shared_representation,
    ) = model(features, training=False)
    
    treatment_rate = tf.reduce_mean(treatments)
    if reweight_sample:
        sample_weight = treatments / (2 * treatment_rate) + (1 - treatments) / (2 * (1 - treatment_rate))
    else:
        sample_weight = tf.ones_like(treatments)
    
    non_randomized_mask = tf.reshape(tf.cast(~tf.cast(randomized_flags, tf.bool), tf.bool), (-1,))
    
    def mask_non_randomized(values):
        return tf.boolean_mask(values, non_randomized_mask)

    losses = {}
    if compute_loss:
        masked_sample_weight = tf.reshape(tf.boolean_mask(tf.reshape(sample_weight, (-1,)), non_randomized_mask), (-1, 1))
        
        if propensity_weight > 0:
            positive_weight = 1.0 / (2.0 * tf.maximum(treatment_rate, 1e-5))
            losses['propensity'] = propensity_weight * weighted_logit_binary_cross_entropy(
                mask_non_randomized(treatments),
                mask_non_randomized(propensity_logit),
                positive_weight,
            )
        if entire_treated_weight > 0:
            losses['estr'] = entire_treated_weight * weighted_binary_cross_entropy(
                mask_non_randomized(outcomes * treatments),
                mask_non_randomized(entire_space_treated_response),
                masked_sample_weight,
            )
        if entire_control_weight > 0:
            losses['escr'] = entire_control_weight * weighted_binary_cross_entropy(
                mask_non_randomized(outcomes * (1 - treatments)),
                mask_non_randomized(entire_space_control_response),
                masked_sample_weight,
            )
        
        treated_mask = treatments[:, 0] > 0.5
        control_mask = ~treated_mask
        if treated_response_weight > 0:
            losses['tr'] = treated_response_weight * masked_binary_cross_entropy(
                outcomes, treated_response_head, treated_mask
            )
        if control_response_weight > 0:
            losses['cr'] = control_response_weight * masked_binary_cross_entropy(
                outcomes, control_response_head, control_mask
            )
        
        if cross_treated_weight > 0:
            cross_treated_response = tf.nn.sigmoid(control_response_logit + pseudo_effect_logit)
            losses['cross_tr'] = cross_treated_weight * masked_binary_cross_entropy(
                outcomes, cross_treated_response, treated_mask
            )
        if cross_control_weight > 0:
            cross_control_response = tf.nn.sigmoid(treated_response_logit - pseudo_effect_logit)
            losses['cross_cr'] = cross_control_weight * masked_binary_cross_entropy(
                outcomes, cross_control_response, control_mask
            )
        
        if compute_imbalance_loss and imbalance_weight > 0:
            imbalance_loss = wasserstein_distance(shared_representation, treatments)
            if imbalance_type == 'mmd':
                imbalance_loss = mmd_distance(shared_representation, treatments)
            losses['imb'] = imbalance_weight * imbalance_loss
    
    total_loss = tf.add_n(list(losses.values())) if losses else tf.constant(0.0)
    
    propensity_predicted = tf.nn.sigmoid(propensity_logit).numpy()
    treatment_effect_predicted = (treated_response_head - control_response_head).numpy()
    outcomes_numpy = outcomes.numpy()
    treatments_numpy = treatments.numpy()
    factual_outcome_predicted = (treated_response_head * treatments + control_response_head * (1 - treatments)).numpy()
    counterfactual_outcome_predicted = (control_response_head * treatments + treated_response_head * (1 - treatments)).numpy()
    
    results = {
        'total_loss': float(total_loss.numpy()),
        'p_prpsy': propensity_predicted,
        'p_yf': factual_outcome_predicted,
        'p_ycf': counterfactual_outcome_predicted,
        'p_tau': treatment_effect_predicted,
    }
    try:
        results['auuc'] = qini_auc_score(
            outcomes_numpy.reshape(-1),
            treatment_effect_predicted.reshape(-1),
            treatments_numpy.reshape(-1),
        )
    except Exception:
        results['auuc'] = 0.0
    
    flattened_treatments = treatments_numpy.flatten()
    if treatment_effect_true is not None:
        treatment_effect_true_numpy = treatment_effect_true.numpy()
        results['sqrt_pehe'] = np.sqrt(np.mean(np.square(treatment_effect_predicted - treatment_effect_true_numpy)))
        results['e_ate'] = np.abs(treatment_effect_predicted.mean() - treatment_effect_true_numpy.mean())
        if np.any(flattened_treatments == 1):
            results['e_att'] = np.abs(
                treatment_effect_predicted[flattened_treatments == 1].mean()
                - treatment_effect_true_numpy[flattened_treatments == 1].mean()
            )
    elif np.any(flattened_treatments == 1) and np.any(flattened_treatments == 0):
        flattened_outcomes = outcomes_numpy.flatten()
        observed_att = flattened_outcomes[flattened_treatments == 1].mean() - flattened_outcomes[flattened_treatments == 0].mean()
        results['e_att'] = np.abs(treatment_effect_predicted[flattened_treatments == 1].mean() - observed_att)
    
    for loss_name, loss_value in losses.items():
        results[f'loss_{loss_name}'] = float(loss_value.numpy())
    return results

print('evaluate() defined.')


evaluate() defined.


### 3.7 Training


In [10]:
from pathlib import Path

def save_test_prediction_artifacts(model_name, experiment_predictions):
    """Save public-code-style test prediction artifacts: units x experiment x output."""
    output_dir = Path(TRAIN_CONFIG['prediction_output_dir'])
    output_dir.mkdir(parents=True, exist_ok=True)

    prediction_arrays = {}
    for prediction_name in ['p_prpsy', 'p_yf', 'p_ycf', 'p_tau']:
        prediction_arrays[prediction_name] = np.stack(
            [experiment_result[prediction_name] for experiment_result in experiment_predictions],
            axis=1,
        )

    loss_by_experiment = np.array([experiment_result['loss'] for experiment_result in experiment_predictions])
    prediction_arrays['loss'] = np.transpose(loss_by_experiment, (1, 2, 0))
    prediction_arrays['val'] = np.array([])

    output_path = output_dir / f'{model_name}_test_result.test'
    np.savez(output_path, **prediction_arrays)
    print(f'Saved test predictions: {output_path}.npz')


def train_step(model, features_batch, treatments_batch, outcomes_batch, randomized_flags_batch, loss_weights, optimizer):
    with tf.GradientTape() as tape:
        (
            propensity_logit,
            entire_space_treated_response,
            entire_space_control_response,
            pseudo_effect_logit,
            treated_response_logit,
            control_response_logit,
            _,
            _,
            _,
            treated_response_head,
            control_response_head,
            shared_representation,
        ) = model(features_batch, training=True)
        
        treatment_rate = tf.reduce_mean(treatments_batch)
        if loss_weights.get('reweight_sample', True):
            sample_weight = treatments_batch / (2 * treatment_rate) + (1 - treatments_batch) / (2 * (1 - treatment_rate))
        else:
            sample_weight = tf.ones_like(treatments_batch)
        
        non_randomized_mask = tf.reshape(tf.cast(~tf.cast(randomized_flags_batch, tf.bool), tf.bool), (-1,))

        def mask_non_randomized(values):
            return tf.boolean_mask(values, non_randomized_mask)

        masked_sample_weight = tf.reshape(tf.boolean_mask(tf.reshape(sample_weight, (-1,)), non_randomized_mask), (-1, 1))
        
        total_loss = tf.constant(0.0)
        if loss_weights['prpsy_w'] > 0:
            positive_weight = 1.0 / (2.0 * tf.maximum(treatment_rate, 1e-5))
            total_loss += loss_weights['prpsy_w'] * weighted_logit_binary_cross_entropy(
                mask_non_randomized(treatments_batch),
                mask_non_randomized(propensity_logit),
                positive_weight,
            )
        if loss_weights['escvr1_w'] > 0:
            total_loss += loss_weights['escvr1_w'] * weighted_binary_cross_entropy(
                mask_non_randomized(outcomes_batch * treatments_batch),
                mask_non_randomized(entire_space_treated_response),
                masked_sample_weight,
            )
        if loss_weights['escvr0_w'] > 0:
            total_loss += loss_weights['escvr0_w'] * weighted_binary_cross_entropy(
                mask_non_randomized(outcomes_batch * (1 - treatments_batch)),
                mask_non_randomized(entire_space_control_response),
                masked_sample_weight,
            )
        
        treated_mask = treatments_batch[:, 0] > 0.5
        control_mask = ~treated_mask
        if loss_weights['h1_w'] > 0:
            total_loss += loss_weights['h1_w'] * masked_binary_cross_entropy(
                outcomes_batch, treated_response_head, treated_mask
            )
        if loss_weights['h0_w'] > 0:
            total_loss += loss_weights['h0_w'] * masked_binary_cross_entropy(
                outcomes_batch, control_response_head, control_mask
            )
        
        if loss_weights['mu1hat_w'] > 0:
            cross_treated_response = tf.nn.sigmoid(control_response_logit + pseudo_effect_logit)
            total_loss += loss_weights['mu1hat_w'] * masked_binary_cross_entropy(
                outcomes_batch, cross_treated_response, treated_mask
            )
        if loss_weights['mu0hat_w'] > 0:
            cross_control_response = tf.nn.sigmoid(treated_response_logit - pseudo_effect_logit)
            total_loss += loss_weights['mu0hat_w'] * masked_binary_cross_entropy(
                outcomes_batch, cross_control_response, control_mask
            )
        
        if loss_weights['imb_dist_w'] > 0:
            imbalance_loss = wasserstein_distance(shared_representation, treatments_batch)
            if loss_weights['imb_dist'] == 'mmd':
                imbalance_loss = mmd_distance(shared_representation, treatments_batch)
            total_loss += loss_weights['imb_dist_w'] * imbalance_loss
    
    gradients = tape.gradient(total_loss, model.trainable_variables)
    grads_and_vars = [(g, v) for g, v in zip(gradients, model.trainable_variables) if g is not None]
    optimizer.apply_gradients(grads_and_vars)
    return total_loss


def train(model, features_train, outcomes_train, treatments_train, randomized_flags_train,
          features_validation, outcomes_validation, treatments_validation, randomized_flags_validation,
          features_test, outcomes_test, treatments_test, randomized_flags_test, treatment_effect_test=None,
          weights=None, epochs=15, batch_size=500, lr=1e-3, l2=1e-2, name='Model'):
    """Train one model. Returns history, selected metrics, and test prediction artifact arrays."""
    loss_weights = dict(weights or {})
    loss_weights.setdefault('reweight_sample', True)
    loss_weights.setdefault('imb_dist', 'wass')
    for weight_name in ['prpsy_w','escvr1_w','escvr0_w','h1_w','h0_w','mu1hat_w','mu0hat_w','imb_dist_w']:
        loss_weights.setdefault(weight_name, 0.0)
    
    train_dataset = tf.data.Dataset.from_tensor_slices(
        (features_train, treatments_train, outcomes_train, randomized_flags_train)
    )
    train_dataset = train_dataset.shuffle(len(features_train)).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr, weight_decay=l2)
    
    history = {'epoch': [], 'lr': [], 'train_loss': [], 'val_loss': [],
               'auuc': [], 'e_att': [], 'sqrt_pehe': [], 'e_ate': []}
    test_prediction_history = {'p_prpsy': [], 'p_yf': [], 'p_ycf': [], 'p_tau': [], 'loss': []}
    
    print(f'[{name}] epochs={epochs}  batch={batch_size}  lr={lr}  l2={l2}')
    print(f'  weights: prpsy={loss_weights["prpsy_w"]} estr={loss_weights["escvr1_w"]} escr={loss_weights["escvr0_w"]} '
          f'h1={loss_weights["h1_w"]} h0={loss_weights["h0_w"]} '
          f'xTR={loss_weights["mu1hat_w"]} xCR={loss_weights["mu0hat_w"]} imb={loss_weights["imb_dist_w"]}')
    print(f'  train: {len(features_train):,}  val: {len(features_validation):,}  test: {len(features_test):,}  '
          f'treated(train): {tf.reduce_mean(treatments_train).numpy():.3f}')
    print('-' * 60)
    
    for epoch_index in tqdm(range(epochs), desc=name, unit="ep"):
        decay_exponent = epoch_index // TRAIN_CONFIG['decay_step_size']
        current_lr = lr * (TRAIN_CONFIG['decay_rate'] ** decay_exponent)
        optimizer.learning_rate.assign(current_lr)
        epoch_losses = []
        for features_batch, treatments_batch, outcomes_batch, randomized_flags_batch in train_dataset:
            if tf.shape(features_batch)[0] < batch_size:
                continue
            batch_loss = train_step(
                model,
                features_batch,
                treatments_batch,
                outcomes_batch,
                randomized_flags_batch,
                loss_weights,
                optimizer,
            )
            epoch_losses.append(float(batch_loss.numpy()))
        
        average_loss = np.mean(epoch_losses) if epoch_losses else 0.0
        validation_results = evaluate(
            model,
            features_validation,
            outcomes_validation,
            treatments_validation,
            randomized_flags_validation,
            weights=loss_weights,
            compute_loss=True,
            compute_imbalance_loss=False,
        )
        test_results = evaluate(
            model,
            features_test,
            outcomes_test,
            treatments_test,
            randomized_flags_test,
            treatment_effect_true=treatment_effect_test,
            weights=loss_weights,
            compute_loss=False,
        )
        
        history['epoch'].append(epoch_index)
        history['lr'].append(float(current_lr))
        history['train_loss'].append(average_loss)
        history['val_loss'].append(validation_results['total_loss'])
        history['auuc'].append(test_results.get('auuc', 0))
        history['e_att'].append(test_results.get('e_att', 0))
        history['sqrt_pehe'].append(test_results.get('sqrt_pehe', 0))
        history['e_ate'].append(test_results.get('e_ate', 0))
        for prediction_name in ['p_prpsy', 'p_yf', 'p_ycf', 'p_tau']:
            test_prediction_history[prediction_name].append(test_results[prediction_name].reshape(-1))
        test_prediction_history['loss'].append([test_results['total_loss']])
        
    
    print('-' * 60)
    if history['auuc']:
        selected_epoch_index = int(np.argmax(history['auuc']))
        selected_test_results = {
            'selection': 'best_auuc',
            'selected_epoch': int(history['epoch'][selected_epoch_index]),
            'auuc': float(history['auuc'][selected_epoch_index]),
            'e_att': float(history['e_att'][selected_epoch_index]),
            'sqrt_pehe': float(history['sqrt_pehe'][selected_epoch_index]),
            'e_ate': float(history['e_ate'][selected_epoch_index]),
        }
    else:
        selected_test_results = {
            'selection': 'best_auuc',
            'selected_epoch': -1,
            'auuc': 0.0,
            'e_att': 0.0,
            'sqrt_pehe': 0.0,
            'e_ate': 0.0,
        }

    print(f'[{name}] Best AUUC epoch={selected_test_results["selected_epoch"] + 1}/{epochs}: '
          f'AUUC={selected_test_results["auuc"]:.4f}  '
          f'e_ATT={selected_test_results["e_att"]:.4f}  '
          f'sqrtPEHE={selected_test_results["sqrt_pehe"]:.4f}  '
          f'e_ATE={selected_test_results["e_ate"]:.4f}')
    
    test_prediction_artifact = {
        prediction_name: np.stack(values, axis=1)
        for prediction_name, values in test_prediction_history.items()
        if prediction_name != 'loss'
    }
    test_prediction_artifact['loss'] = np.array(test_prediction_history['loss'])

    return history, selected_test_results, test_prediction_artifact

print('train() defined.')


train() defined.


---


## 4. Model Training

Each model uses the same `DESCN` backbone. Only the loss weights differ (Section 2.3).

The paper has two separate counters:

- `EXPERIMENT_COUNT = 5`: five independent experiment subsets/repeats used to report mean +/- standard error.
- `TRAIN_CONFIG['epochs'] = 5`: five epochs inside each one of those experiments for the Production dataset.

So each model is trained five independent times, and each independent training runs for five epochs. The randomized Lazada test CSV is reused across experiments.


### 4.1 TARNet [[Shalit et al., 2017]](https://arxiv.org/abs/1606.03976)

TARNet uses a shared representation with two heads (treated/control response), trained only on their respective sub-spaces. No entire space learning, no cross connections, no IPM regularization.


In [11]:
def train_tarnet():
    """Train TARNet model. Returns (results, histories, sqrt_pehe, e_ate, auuc)."""
    seed_everything(2)
    loss_weights = MODEL_CONFIGS['TARNet']

    tarnet_results = []
    histories = []
    prediction_artifacts = []

    for experiment_index in range(EXPERIMENT_COUNT):
        experiment_data = make_experiment_tensors(experiment_index)
        tarnet_model = DESCN(
            input_dim=training_data['x'].shape[1],
            share_dim=HYPERPARAMS['share_dim'],
            base_dim=HYPERPARAMS['base_dim'],
            dropout=HYPERPARAMS['dropout'],
            use_batchnorm=HYPERPARAMS['use_batchnorm'],
        )
        _ = tarnet_model(tf.zeros((2, training_data['x'].shape[1])), training=False)
        
        history, selected_metrics, test_prediction_artifact = train(
            tarnet_model,
            experiment_data['features_train'],
            experiment_data['outcomes_train'],
            experiment_data['treatments_train'],
            experiment_data['randomized_flags_train'],
            experiment_data['features_validation'],
            experiment_data['outcomes_validation'],
            experiment_data['treatments_validation'],
            experiment_data['randomized_flags_validation'],
            experiment_data['features_test'],
            experiment_data['outcomes_test'],
            experiment_data['treatments_test'],
            experiment_data['randomized_flags_test'],
            experiment_data['treatment_effect_test'],
            weights=loss_weights,
            epochs=TRAIN_CONFIG['epochs'],
            batch_size=HYPERPARAMS['batch_size'],
            lr=HYPERPARAMS['lr'],
            l2=HYPERPARAMS['l2'],
            name=f'TARNet exp {experiment_index + 1}/{EXPERIMENT_COUNT}',
        )
        tarnet_results.append(selected_metrics)
        histories.append(history)
        prediction_artifacts.append(test_prediction_artifact)
        del tarnet_model, experiment_data
        tf.keras.backend.clear_session()
        gc.collect()

    save_test_prediction_artifacts('TARNet_tf', prediction_artifacts)

    sqrt_pehe_values = np.array([0.0])
    e_ate_values = np.array([0.0])
    auuc_values = np.array([result['auuc'] for result in tarnet_results])
    e_att_values = np.array([result.get('e_att', 0.0) for result in tarnet_results])
    print(f'TARNet ({EXPERIMENT_COUNT} experiments): AUUC={auuc_values.mean():.4f}+/-{auuc_values.std()/EXPERIMENT_COUNT**0.5:.4f}  e_ATT={e_att_values.mean():.4f}+/-{e_att_values.std()/EXPERIMENT_COUNT**0.5:.4f}')
    return tarnet_results, histories, sqrt_pehe_values, e_ate_values, auuc_values, e_att_values


### 4.2 CFR (MMD) [[Shalit et al., 2017]](https://arxiv.org/abs/1606.03976)

CFR extends TARNet with an **IPM regularization** (MMD) on the shared representation to force treated and control distributions closer. This helps reduce treatment bias but does not address sample imbalance.


In [12]:
def train_cfr_mmd():
    """Train CFR(MMD) model. Returns (results, histories, sqrt_pehe, e_ate, auuc)."""
    seed_everything(2)
    loss_weights = MODEL_CONFIGS['CFR_MMD']

    cfr_mmd_results = []
    histories = []
    prediction_artifacts = []

    for experiment_index in range(EXPERIMENT_COUNT):
        experiment_data = make_experiment_tensors(experiment_index)
        cfr_mmd_model = DESCN(
            input_dim=training_data['x'].shape[1],
            share_dim=HYPERPARAMS['share_dim'],
            base_dim=HYPERPARAMS['base_dim'],
            dropout=HYPERPARAMS['dropout'],
            use_batchnorm=HYPERPARAMS['use_batchnorm'],
        )
        _ = cfr_mmd_model(tf.zeros((2, training_data['x'].shape[1])), training=False)
        
        history, selected_metrics, test_prediction_artifact = train(
            cfr_mmd_model,
            experiment_data['features_train'],
            experiment_data['outcomes_train'],
            experiment_data['treatments_train'],
            experiment_data['randomized_flags_train'],
            experiment_data['features_validation'],
            experiment_data['outcomes_validation'],
            experiment_data['treatments_validation'],
            experiment_data['randomized_flags_validation'],
            experiment_data['features_test'],
            experiment_data['outcomes_test'],
            experiment_data['treatments_test'],
            experiment_data['randomized_flags_test'],
            experiment_data['treatment_effect_test'],
            weights=loss_weights,
            epochs=TRAIN_CONFIG['epochs'],
            batch_size=HYPERPARAMS['batch_size'],
            lr=HYPERPARAMS['lr'],
            l2=HYPERPARAMS['l2'],
            name=f'CFR(MMD) exp {experiment_index + 1}/{EXPERIMENT_COUNT}',
        )
        cfr_mmd_results.append(selected_metrics)
        histories.append(history)
        prediction_artifacts.append(test_prediction_artifact)
        del cfr_mmd_model, experiment_data
        tf.keras.backend.clear_session()
        gc.collect()

    save_test_prediction_artifacts('CFR_mmd_tf', prediction_artifacts)

    sqrt_pehe_values = np.array([0.0])
    e_ate_values = np.array([0.0])
    auuc_values = np.array([result['auuc'] for result in cfr_mmd_results])
    e_att_values = np.array([result.get('e_att', 0.0) for result in cfr_mmd_results])
    print(f'CFR(MMD) ({EXPERIMENT_COUNT} experiments): AUUC={auuc_values.mean():.4f}+/-{auuc_values.std()/EXPERIMENT_COUNT**0.5:.4f}  e_ATT={e_att_values.mean():.4f}+/-{e_att_values.std()/EXPERIMENT_COUNT**0.5:.4f}')
    return cfr_mmd_results, histories, sqrt_pehe_values, e_ate_values, auuc_values, e_att_values


### 4.3 X-network

X-network adds the **Pseudo Treatment Effect** $\tau'$ and cross connections between TR and CR. This addresses sample imbalance by allowing both response functions to learn from each other through the PTE bridge. No entire space learning yet.


In [13]:
def train_xnetwork():
    """Train X-network model. Returns (results, histories, sqrt_pehe, e_ate, auuc)."""
    seed_everything(2)
    loss_weights = MODEL_CONFIGS['X_network']

    xnetwork_results = []
    histories = []
    prediction_artifacts = []

    for experiment_index in range(EXPERIMENT_COUNT):
        experiment_data = make_experiment_tensors(experiment_index)
        xnetwork_model = DESCN(
            input_dim=training_data['x'].shape[1],
            share_dim=HYPERPARAMS['share_dim'],
            base_dim=HYPERPARAMS['base_dim'],
            dropout=HYPERPARAMS['dropout'],
            use_batchnorm=HYPERPARAMS['use_batchnorm'],
        )
        _ = xnetwork_model(tf.zeros((2, training_data['x'].shape[1])), training=False)
        
        history, selected_metrics, test_prediction_artifact = train(
            xnetwork_model,
            experiment_data['features_train'],
            experiment_data['outcomes_train'],
            experiment_data['treatments_train'],
            experiment_data['randomized_flags_train'],
            experiment_data['features_validation'],
            experiment_data['outcomes_validation'],
            experiment_data['treatments_validation'],
            experiment_data['randomized_flags_validation'],
            experiment_data['features_test'],
            experiment_data['outcomes_test'],
            experiment_data['treatments_test'],
            experiment_data['randomized_flags_test'],
            experiment_data['treatment_effect_test'],
            weights=loss_weights,
            epochs=TRAIN_CONFIG['epochs'],
            batch_size=HYPERPARAMS['batch_size'],
            lr=HYPERPARAMS['lr'],
            l2=HYPERPARAMS['l2'],
            name=f'X-network exp {experiment_index + 1}/{EXPERIMENT_COUNT}',
        )
        xnetwork_results.append(selected_metrics)
        histories.append(history)
        prediction_artifacts.append(test_prediction_artifact)
        del xnetwork_model, experiment_data
        tf.keras.backend.clear_session()
        gc.collect()

    save_test_prediction_artifacts('Xnetwork_tf', prediction_artifacts)

    sqrt_pehe_values = np.array([0.0])
    e_ate_values = np.array([0.0])
    auuc_values = np.array([result['auuc'] for result in xnetwork_results])
    e_att_values = np.array([result.get('e_att', 0.0) for result in xnetwork_results])
    print(f'X-network ({EXPERIMENT_COUNT} experiments): AUUC={auuc_values.mean():.4f}+/-{auuc_values.std()/EXPERIMENT_COUNT**0.5:.4f}  e_ATT={e_att_values.mean():.4f}+/-{e_att_values.std()/EXPERIMENT_COUNT**0.5:.4f}')
    return xnetwork_results, histories, sqrt_pehe_values, e_ate_values, auuc_values, e_att_values


### 4.4 DESCN

DESCN combines **ESN** (entire space learning via $\text{ESTR} = \mu_1\pi$ and $\text{ESCR} = \mu_0(1-\pi)$) with **X-network** (cross connections via $\tau'$). TR and CR are trained only through ESTR/ESCR in the entire space ($h_1, h_0$ weights = 0). This addresses both treatment bias and sample imbalance simultaneously.


In [14]:
def train_descn():
    """Train DESCN model. Returns (results, histories, sqrt_pehe, e_ate, auuc)."""
    seed_everything(2)
    loss_weights = MODEL_CONFIGS['DESCN']

    descn_results = []
    histories = []
    prediction_artifacts = []

    for experiment_index in range(EXPERIMENT_COUNT):
        experiment_data = make_experiment_tensors(experiment_index)
        descn_model = DESCN(
            input_dim=training_data['x'].shape[1],
            share_dim=HYPERPARAMS['share_dim'],
            base_dim=HYPERPARAMS['base_dim'],
            dropout=HYPERPARAMS['dropout'],
            use_batchnorm=HYPERPARAMS['use_batchnorm'],
        )
        _ = descn_model(tf.zeros((2, training_data['x'].shape[1])), training=False)
        
        history, selected_metrics, test_prediction_artifact = train(
            descn_model,
            experiment_data['features_train'],
            experiment_data['outcomes_train'],
            experiment_data['treatments_train'],
            experiment_data['randomized_flags_train'],
            experiment_data['features_validation'],
            experiment_data['outcomes_validation'],
            experiment_data['treatments_validation'],
            experiment_data['randomized_flags_validation'],
            experiment_data['features_test'],
            experiment_data['outcomes_test'],
            experiment_data['treatments_test'],
            experiment_data['randomized_flags_test'],
            experiment_data['treatment_effect_test'],
            weights=loss_weights,
            epochs=TRAIN_CONFIG['epochs'],
            batch_size=HYPERPARAMS['batch_size'],
            lr=HYPERPARAMS['lr'],
            l2=HYPERPARAMS['l2'],
            name=f'DESCN exp {experiment_index + 1}/{EXPERIMENT_COUNT}',
        )
        descn_results.append(selected_metrics)
        histories.append(history)
        prediction_artifacts.append(test_prediction_artifact)
        del descn_model, experiment_data
        tf.keras.backend.clear_session()
        gc.collect()

    save_test_prediction_artifacts('DESCN_tf', prediction_artifacts)

    sqrt_pehe_values = np.array([0.0])
    e_ate_values = np.array([0.0])
    auuc_values = np.array([result['auuc'] for result in descn_results])
    e_att_values = np.array([result.get('e_att', 0.0) for result in descn_results])
    print(f'DESCN ({EXPERIMENT_COUNT} experiments): AUUC={auuc_values.mean():.4f}+/-{auuc_values.std()/EXPERIMENT_COUNT**0.5:.4f}  e_ATT={e_att_values.mean():.4f}+/-{e_att_values.std()/EXPERIMENT_COUNT**0.5:.4f}')
    return descn_results, histories, sqrt_pehe_values, e_ate_values, auuc_values, e_att_values


---


## 5. Results Comparison

Reproducing the Production-Dataset side of Table 2 from the paper/public code: model performance is evaluated on the randomized Lazada test set with AUUC and ATT error. The public CSV is smaller than the full production table in the paper, so the values are not expected to exactly match KDD Table 2.

The original public evaluation script (`eval4real_data.py`) scans saved epoch outputs and reports the epoch with the best AUUC for each experiment. This notebook mirrors that behavior by selecting each experiment's best-AUUC epoch from the per-epoch test metrics, instead of reporting only the final epoch. It also saves public-code-style test prediction artifacts under `results/lzd_real_tf/` with keys `p_prpsy`, `p_yf`, `p_ycf`, `p_tau`, `loss`, and `val`.


In [ ]:
import matplotlib.pyplot as plt

# Train all models
tarnet_results, tarnet_histories, _, _, tarnet_auuc_values, tarnet_e_att_values = train_tarnet()
cfr_mmd_results, cfr_mmd_histories, _, _, cfr_mmd_auuc_values, cfr_mmd_e_att_values = train_cfr_mmd()
xnetwork_results, xnetwork_histories, _, _, xnetwork_auuc_values, xnetwork_e_att_values = train_xnetwork()
descn_results, descn_histories, _, _, descn_auuc_values, descn_e_att_values = train_descn()

baseline_model_name = 'CFR(MMD)'
model_names = ['TARNet', baseline_model_name, 'X-network', 'DESCN']
auuc_values_by_model = [
    tarnet_auuc_values,
    cfr_mmd_auuc_values,
    xnetwork_auuc_values,
    descn_auuc_values,
]
e_att_values_by_model = [
    tarnet_e_att_values,
    cfr_mmd_e_att_values,
    xnetwork_e_att_values,
    descn_e_att_values,
]

baseline_auuc = cfr_mmd_auuc_values.mean()
baseline_e_att = cfr_mmd_e_att_values.mean()

print('=' * 96)
print(f'MODEL COMPARISON - Lazada Production ({EXPERIMENT_COUNT} experiments, best AUUC epoch, mean +/- std error)')
print('=' * 96)
print(f'{"Model":<16s} {"AUUC":<22s} {"AUUC Impr%":>11s} {"e_ATT":<22s} {"e_ATT Impr%":>12s}')
print('-' * 96)

for model_name, auuc_values, e_att_values in zip(model_names, auuc_values_by_model, e_att_values_by_model):
    auuc_mean = auuc_values.mean()
    auuc_se = auuc_values.std() / EXPERIMENT_COUNT**0.5
    e_att_mean = e_att_values.mean()
    e_att_se = e_att_values.std() / EXPERIMENT_COUNT**0.5
    auuc_impr = (auuc_mean - baseline_auuc) / baseline_auuc * 100 if baseline_auuc != 0 else 0
    e_att_impr = (baseline_e_att - e_att_mean) / baseline_e_att * 100 if baseline_e_att != 0 else 0
    best = '  <<' if model_name == 'DESCN' else ''
    print(
        f'{model_name:<16s} '
        f'{auuc_mean:.4f} +/- {auuc_se:.4f}     '
        f'{auuc_impr:>+8.1f}% '
        f'{e_att_mean:.4f} +/- {e_att_se:.4f}     '
        f'{e_att_impr:>+8.1f}%{best}'
    )

print()
print(f'Improvement over {baseline_model_name} baseline.')
print('AUUC: higher is better. e_ATT: lower is better.')

# Plot AUUC, matching the main uplift-ranking metric for Production.
fig, ax = plt.subplots(figsize=(8, 4))
bar_colors = ['#2196F3', '#FF9800', '#4CAF50', '#F44336']
means = [v.mean() for v in auuc_values_by_model]
errors = [v.std() / EXPERIMENT_COUNT**0.5 for v in auuc_values_by_model]
ax.bar(model_names, means, yerr=errors, color=bar_colors, capsize=5)
ax.set_title('AUUC - Lazada Production')
ax.grid(axis='y', alpha=0.3)
for i, value in enumerate(means):
    ax.text(i, value + errors[i], f'{value:.4f}', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig('model_comparison_lazada.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved to model_comparison_lazada.png')


[TARNet exp 1/5] epochs=5  batch=5000  lr=0.001  l2=0.001
  weights: prpsy=0 estr=0 escr=0 h1=1 h0=1 xTR=0 xCR=0 imb=0
  train: 667,203  val: 166,800  test: 181,669  treated(train): 0.222
------------------------------------------------------------


TARNet exp 1/5: 100%|██████████| 5/5 [02:06<00:00, 25.26s/ep]


------------------------------------------------------------
[TARNet exp 1/5] Best AUUC epoch=5/5: AUUC=0.0239  e_ATT=0.0158  sqrtPEHE=0.0000  e_ATE=0.0000
[TARNet exp 2/5] epochs=5  batch=5000  lr=0.001  l2=0.001
  weights: prpsy=0 estr=0 escr=0 h1=1 h0=1 xTR=0 xCR=0 imb=0
  train: 667,203  val: 166,800  test: 181,669  treated(train): 0.222
------------------------------------------------------------


TARNet exp 2/5: 100%|██████████| 5/5 [02:12<00:00, 26.41s/ep]


------------------------------------------------------------
[TARNet exp 2/5] Best AUUC epoch=3/5: AUUC=0.0240  e_ATT=0.0121  sqrtPEHE=0.0000  e_ATE=0.0000
[TARNet exp 3/5] epochs=5  batch=5000  lr=0.001  l2=0.001
  weights: prpsy=0 estr=0 escr=0 h1=1 h0=1 xTR=0 xCR=0 imb=0
  train: 667,203  val: 166,800  test: 181,669  treated(train): 0.222
------------------------------------------------------------


TARNet exp 3/5:   0%|          | 0/5 [00:00<?, ?ep/s]

In [ ]:
# Release GPU VRAM (useful between long runs on Apple Metal / CUDA)
import gc
tf.keras.backend.clear_session()
gc.collect()
print('VRAM released.')

---


## 6. Summary

### What DESCN achieves

1. **ESN** learns response functions in the **entire sample space** via $\text{ESTR} = \mu_1\pi$ and $\text{ESCR} = \mu_0(1-\pi)$. A treated sample teaches control outcomes, and vice versa. Implicitly performs Inverse Probability Weighting.

2. **X-network** bridges treated and control responses through a **Pseudo Treatment Effect** $\tau'$, operating in logit space for numerical stability. Cross predictions create counterfactual estimates that balance learning when one group is much smaller than the other.

3. **Unified architecture**: TARNet, CFR, X-network, and DESCN all use the same backbone. Only loss weights differ, making the framework a flexible testbed for uplift modeling.

### References

- **Zhong et al. (2022)**. *DESCN: Deep Entire Space Cross Networks for Individual Treatment Effect Estimation*. KDD '22. [arXiv:2207.09920](https://arxiv.org/abs/2207.09920)
- **Shalit et al. (2017)**. *Estimating individual treatment effect: generalization bounds and algorithms*. ICML '17. (TARNet/CFR)
- **Kunzel et al. (2019)**. *Metalearners for estimating heterogeneous treatment effects using machine learning*. PNAS. (X-learner)
- **Rosenbaum & Rubin (1983)**. *The central role of the propensity score in observational studies for causal effects*. Biometrika. (Propensity Score / IPW)
- **Original DESCN code**: [github.com/kailiang-zhong/DESCN](https://github.com/kailiang-zhong/DESCN)
